**Install** **&** **imports**

In [10]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

**Task 4.1: One validated call**

In [ ]:
MODEL_NAME = "gemini-3.6-flash"

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = "You are a clinic operations assistant returning structured JSON."


def build_craft_prompt(note_text):
    return (
        "Context: CareLoop Clinics runs outpatient clinics across India, and call-centre staff log "
        "short notes after calling patients who missed an appointment.\n"
        "Role: You are an outpatient clinic operations analyst categorizing missed-visit reasons.\n"
        "Action: Read the follow-up note below and extract: "
        "(1) root_cause — choose exactly ONE value from this list: "
        "forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, "
        "work_or_schedule, other. If the note is vague, ambiguous, or does not clearly state why "
        "the patient missed (e.g., phone switched off or unreachable), output 'other' rather than guessing; "
        "(2) confidence — a float between 0 and 1; "
        "(3) summary — a concise summary in at most 25 words; "
        "(4) recommended_action — a practical next step for the front desk.\n"
        "Tone: Objective, concise, and operational.\n"
        "Format: Valid JSON matching the requested schema.\n\n"
        f"Note: {note_text}"
    )


API_KEY = input("Paste Your Key: ")
client = genai.Client(api_key=API_KEY)
api_call_count = 0


def analyse_note(note_text, prompt_fn=build_craft_prompt, return_raw=False):
    global api_call_count
    prompt = prompt_fn(note_text)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    api_call_count += 1
    raw_text = response.text
    data = json.loads(raw_text)
    if return_raw:
        return data, raw_text
    return data


notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_result = analyse_note(n001_text)
print("N001 parsed dictionary:")
print(n001_result)

**Task 4.2: Validate and process the first 10 notes**

In [12]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


results = []
first_10 = notes_df.head(10)

for i, row in first_10.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        print(f"{note_id}: skipped_blank (no API call)")
        continue

    if api_call_count > 0:
        time.sleep(13)

    raw_text = ""
    data = {}
    status = "invalid"

    try:
        data, raw_text = analyse_note(str(note_text), return_raw=True)
        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_CAUSES:
            status = "valid"
    except Exception as e:
        print(f"{note_id}: validation/API issue -> {e}")

    results.append({
        "note_id": note_id,
        "appointment_id": appt_id,
        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_text,
    })
    print(f"{note_id}: {status} -> {data.get('root_cause', '')}")

analysed_df = pd.DataFrame(results, columns=[
    "note_id", "appointment_id", "root_cause", "confidence",
    "summary", "recommended_action", "validation_status", "raw_response"
])
analysed_df.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv with shape:", analysed_df.shape)

N001: valid -> forgot_or_no_reminder
N002: valid -> distance_or_transport
N003: valid -> cost_or_fee
N004: skipped_blank (no API call)
N005: valid -> illness_or_emergency
N006: valid -> work_or_schedule
N007: valid -> forgot_or_no_reminder
N008: valid -> distance_or_transport
N009: valid -> other
N010: valid -> work_or_schedule

Saved notes_analysed.csv with shape: (10, 8)


**Task 4.3: Break it on purpose & compare** **prompts**

In [13]:
print("=== (a) Deliberate Pydantic validation failures ===")
bad_confidence = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Patient missed visit",
    "recommended_action": "Re-call patient",
}
bad_missing_field = {
    "root_cause": "other",
    "confidence": 0.8,
    "summary": "Patient missed visit",
}

for label, payload in [("confidence=1.7", bad_confidence), ("missing recommended_action", bad_missing_field)]:
    try:
        FollowupInsight(**payload)
    except ValidationError as e:
        print(f"\nError for {label}:\n{e}")

print("\n=== (b) Prompt V1 vs V2 on N007 and N009 ===")


def build_v1_prompt(note_text):
    return f"Classify this note: {note_text}"


comparison_rows = []
for target_id in ["N007", "N009"]:
    text = notes_df.loc[notes_df["note_id"] == target_id, "note_text"].iloc[0]
    for version, fn in [("V1", build_v1_prompt), ("V2", build_craft_prompt)]:
        time.sleep(13)
        res = analyse_note(text, prompt_fn=fn)
        rc = res.get("root_cause", "")
        comparison_rows.append({
            "note_id": target_id,
            "prompt": version,
            "root_cause": rc,
            "in_allowed_list": rc in ALLOWED_CAUSES,
            "confidence": res.get("confidence", ""),
        })

comp_df = pd.DataFrame(comparison_rows)
print(comp_df.to_string(index=False))

print(f"\n=== (c) Total Gemini API calls used across Tasks 4.1-4.3: {api_call_count} ===")

=== (a) Deliberate Pydantic validation failures ===

Error for confidence=1.7:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Error for missing recommended_action:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '... 'Patient missed visit'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

=== (b) Prompt V1 vs V2 on N007 and N009 ===
note_id prompt                     root_cause  in_allowed_list  confidence
   N007     V1 Missed SMS / Communication Gap            False        0.92
   N007     V2          forgot_or_no_reminder             True        0.95
   N009     V1            Unreachable Patient            False        0.95
   N009     V2                          other      

 I would deploy Prompt V2 (CRAFT) because V1 invented out-of-schema labels (False), whereas V2 reliably produced valid allowed root_cause categories—including other for the ambiguous note N009 and forgot_or_no_reminder for the Hinglish note N007.

**Task 4.4: Merge & cross-tabulate**

In [14]:
appts_clean = pd.read_csv("appointments_clean.csv")
notes_analysed = pd.read_csv("notes_analysed.csv")

merged_notes = notes_analysed.merge(appts_clean, on="appointment_id", how="left")
print("Merged table shape:", merged_notes.shape)

ct = pd.crosstab(merged_notes["root_cause"], merged_notes["reminder_sent"], margins=True)
print("\nCross-tabulation of root_cause vs reminder_sent:")
print(ct)

Merged table shape: (10, 23)

Cross-tabulation of root_cause vs reminder_sent:
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10
